In [ ]:
import os
import numpy as np
import pickle
import torch
from argparse import ArgumentParser
from tqdm import tqdm
import glob

from articulate.model import ParametricModel
from articulate import math
from config import paths, datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn as nn
from config import *
from helpers import * 
from data import PoseDataset
import torch
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:

# dataset_name = "dip"
# dataset = PoseDataset( evaluate=dataset_name)   # loads processed_datasets

# dataset = PoseDataset(fold='test', evaluate=dataset_name)   # loads processed_datasets
dataset = PoseDataset(fold='train', finetune=None)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


In [ ]:
def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)
def _dataloader(dataset):
        return DataLoader(
            dataset, 
            batch_size=32, 
            collate_fn=pad_seq, 
            num_workers=0, 
            shuffle=True, 
            drop_last=True
        )

In [ ]:
train_dataloader_ = _dataloader(train_dataset)

In [ ]:
len(train_dataloader_)

In [ ]:
all_pose_trans = []

for (inputs, input_lengths), (outputs, output_lengths) in train_dataloader_:

    poses = outputs["poses"]   # (B, S, 144)
    trans = outputs["trans"]   # (B, S, 3)

    
    pose_trans = torch.cat([poses, trans], dim=-1)  # (B, S, 147)

    all_pose_trans.append(pose_trans)


In [ ]:
class TemporalDenoiser(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        
        #encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        #decodeer
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x, t):
        B, T, D = x.shape
        
        # Encoder
        x_encoded = self.encoder(x)  # [B, T, hidden_dim]
        
        # Time embedding
        t_emb = self.time_mlp(t)  # [B, hidden_dim]
        t_expanded = t_emb.unsqueeze(1).expand(B, T, -1)  # [B, T, hidden_dim]
        t_expanded = t_expanded.to(x.device)
        
        # Concatenate encoded input + time
        x_in = torch.cat([x_encoded, t_expanded], dim=-1)
        x_flat = x_in.view(B * T, -1)
        
        # Denoise
        out = self.mlp(x_flat)
        
        return out.view(B, T, D)

In [ ]:

padded = []
x = all_pose_trans 
max_L = max(x[i].shape[1] for i in range(len(train_dataloader_)))
for i in range(len(train_dataloader_)):
    seq = x[i]  # (B, L_i, 147)
    L_i = seq.shape[1]

    pad_len = max_L - L_i

    padded_seq = F.pad(seq, (0,0,0,pad_len))  # pad time dimension
    padded.append(padded_seq)

x = torch.stack(padded, dim=0)  # (6, B, max_L, 147)


x = x.reshape(-1, max_L, 147)  # (6B, max_L,147)

In [ ]:
x.shape

In [ ]:
import torch
import torch.nn.functional as F

device = "cuda" 

# =====================================
# Hyperparameters
# =====================================
TIMESTEPS = 1000
LR = 1e-4
input_dim = 147  # poses (144) + trans (3)
model = TemporalDenoiser(input_dim=input_dim, hidden_dim=128).to(device)
betas = torch.linspace(1e-4, 0.02, TIMESTEPS)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

alpha_bar = alpha_bar.to(device)


# =====================================
# Noise Function (Sequence Version)
# =====================================
def add_noise_sequence(x0, t):
    """
    x0: (B, L, 147)
    t : (B,)
    """

    device = x0.device  #  get device from input

    noise = torch.randn_like(x0).to(device)

    alpha_t = alpha_bar[t].view(-1, 1, 1).to(device)

    noisy = torch.sqrt(alpha_t) * x0 + torch.sqrt(1 - alpha_t) * noise

    return noisy, noise

# =====================================
# Training Step
# =====================================
def train_step(model, optimizer, x0):
    """
    x0: (B, L, 147)
    """

    B = x0.shape[0]

    # Sample random timesteps per sequence
    t = torch.randint(0, TIMESTEPS, (B,), device=device)

    # Normalize timestep to embedding scale
    t_emb = t.float().unsqueeze(1) / TIMESTEPS
    t_emb = t_emb.to(device)  # Move to device

    # Add noise
    noisy_x, noise = add_noise_sequence(x0, t)

    # Forward pass
    pred_noise = model(noisy_x, t_emb)

    # Noise prediction loss
    loss = F.mse_loss(pred_noise, noise)

    optimizer.zero_grad()
    loss.backward()

    # Optional: gradient clipping (IMPORTANT for long sequences)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    return loss.item()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
x = x.to(device)  # Move data to CUDA
# train_step(model, optimizer, x)

In [ ]:
model

In [ ]:
from tqdm import tqdm

num_epochs = 100

for epoch in range(num_epochs):
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(range(0, x.shape[0], 32), desc=f"Epoch {epoch+1}/{num_epochs}")
    # x_batch has shape (32, max_L, 147)
    for batch_idx in pbar:
        x_batch = x[batch_idx:batch_idx+32]
        loss = train_step(model, optimizer, x_batch)
        total_loss += loss
        num_batches += 1
        
        pbar.set_postfix({'loss': f'{loss:.6f}'})
    
    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.6f}\n")
torch.save(model.state_dict(), "temporal_denoiser.pth")

print("Training complete!")

In [ ]:
gnd_poses = x

In [ ]:
gnd_poses.shape

In [ ]:
from articulate.evaluator import MeanPerJointErrorEvaluator,RotationRepresentation
from articulate.evaluator import to_rotation_matrix

In [ ]:
eval_armature = MeanPerJointErrorEvaluator(
    official_model_file="smpl/basicmodel_m.pkl",
    rep=RotationRepresentation.R6D,    # match your pose format
    device=device)

In [ ]:
B_eval = 4                
L_eval = 24              

gnd_subset = x[:B_eval, :L_eval, :144]   # [B_eval, L_eval, 144]
gnd_poses = gnd_subset.clone().to(device)  # keep on device for later

# now prepare noise of the same shape for reverse diffusion
x = torch.randn(B_eval, L_eval, 147, device=device)  


with torch.no_grad():
    for i in tqdm(reversed(range(TIMESTEPS)), total=TIMESTEPS,
                  desc="Denoising"):
        t = torch.full((B_eval, 1), float(i), device=device,
                       dtype=torch.float) / TIMESTEPS
        noise_pred = model(x, t)
        a = alphas[i]; ab = alpha_bar[i]; b = betas[i]
        z = torch.randn_like(x) if i > 0 else 0
        x = (1/torch.sqrt(a)) * (x - ((1 - a)/(torch.sqrt(1 - ab))) *
                                  noise_pred) + torch.sqrt(b) * z

# ---- now x has shape [B_eval, L_eval, 147] and gnd_poses matches ----
B, L, D = x.shape           # B=B_eval, L=L_eval, D=147

# determine joint count from feature dimension (last dim should be joints*6)
feat = x[..., :144]   # predicted pose features
joint_count = feat.shape[-1] // 6

pred_pose_r6d = feat.view(B * L, joint_count, 6)
# ground truth may have same number of joints; enforce consistency
gt_pose_r6d   = gnd_poses.view(B * L, joint_count, 6)

# make sure they’re contiguous
pred_pose_r6d = pred_pose_r6d.contiguous()
gt_pose_r6d   = gt_pose_r6d.contiguous()

# directly evaluate on 6D representation; evaluator will convert internally
err = eval_armature(pred_pose_r6d.to(device), gt_pose_r6d.to(device))   # returns [pos, local, global]
mpjpe = err[0].item()
print(f"MPJPE = {mpjpe:.4f} (meters)")

# for FullMotionEvaluator you’d index [0,0] instead of [0]

# pose windowing

In [2]:
import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from typing import List
import random
import lightning as L
from tqdm import tqdm 

import articulate as art
from config import *
from utils import *
from helpers import *

In [6]:
class PoseDataset(Dataset):
    def __init__(self, fold: str='train', evaluate: str=None, finetune: str=None):
        super().__init__()
        self.fold = fold
        self.evaluate = evaluate
        self.finetune = finetune
        self.bodymodel = art.model.ParametricModel(paths.smpl_file)
        self.combos = list(amass.combos.items())
        self.data = self._prepare_dataset()

    def _get_data_files(self, data_folder):
        if self.fold == 'train':
            return self._get_train_files(data_folder)
        elif self.fold == 'test':
            return self._get_test_files()
        else:
            raise ValueError(f"Unknown data fold: {self.fold}.")

    def _get_train_files(self, data_folder):
        if self.finetune:
            return [datasets.finetune_datasets[self.finetune]]
        else:
            return [x.name for x in data_folder.iterdir() if not x.is_dir()]

    def _get_test_files(self):
        return [datasets.test_datasets[self.evaluate]]

    def _prepare_dataset(self):
        data_folder = paths.processed_datasets / ('eval' if (self.finetune or self.evaluate) else '')
        data_files = self._get_data_files(data_folder)

        print(f"\n{'='*60}")
        print(f"Loading datasets for {self.fold.upper()} mode")
        print(f"Datasets: {data_files}")
        print(f"{'='*60}\n")

        data = {key: [] for key in [
            'imu_inputs', 'pose_outputs', 'joint_outputs',
            'tran_outputs', 'vel_outputs', 'foot_outputs'
        ]}

        for data_file in tqdm(data_files):
            try:
                file_data = torch.load(data_folder / data_file)
                self._process_file_data(file_data, data)
            except Exception as e:
                print(f"Error processing {data_file}: {e}")

        print("\nTotal pose windows stored:", len(data['pose_outputs']))
        return data

    def _process_file_data(self, file_data, data):
        accs, oris, poses, trans = file_data['acc'], file_data['ori'], file_data['pose'], file_data['tran']
        joints = file_data.get('joint', [None] * len(poses))
        foots = file_data.get('contact', [None] * len(poses))

        for acc, ori, pose, tran, joint, foot in zip(accs, oris, poses, trans, joints, foots):

            print("\n" + "-"*50)
            print("NEW SEQUENCE")
            print("Original pose shape from file:", pose.shape)

            acc, ori = acc[:, :5]/amass.acc_scale, ori[:, :5]

            pose_global, joint = self.bodymodel.forward_kinematics(
                pose=pose.view(-1, 216)
            )

            pose = pose if self.evaluate else pose_global.view(-1, 24, 3, 3)
            joint = joint.view(-1, 24, 3)

            print("Pose after forward kinematics:", pose.shape)

            self._process_combo_data(acc, ori, pose, joint, tran, foot, data)

    def _process_combo_data(self, acc, ori, pose, joint, tran, foot, data):

        for combo_name, c in self.combos:

            print("\n" + "="*50)
            print("Processing combo:", combo_name)

            combo_acc = torch.zeros_like(acc)
            combo_ori = torch.zeros_like(ori)
            combo_acc[:, c] = acc[:, c]
            combo_ori[:, c] = ori[:, c]

            imu_input = torch.cat([combo_acc.flatten(1), combo_ori.flatten(1)], dim=1)

            data_len = len(imu_input) if self.evaluate else datasets.window_length

            # print("IMU input shape:", imu_input.shape)
            # print("Window length used:", data_len)
            # print("Pose shape BEFORE windowing:", pose.shape)

            # splits = torch.split(pose, data_len)
            # splits = torch.split(pose, data_len)
            # splits = splits[:-1] if splits[-1].shape[0] < data_len else splits   #added to have only similar length windows

            # print("Number of pose windows created:", len(splits))
            # print("First pose window shape:", splits[0].shape)
            # print("Last pose window shape:", splits[-1].shape)

            # if len(splits) > 1:
            #     overlap = torch.allclose(splits[0][-1], splits[1][0])
            #     print("Do windows overlap?")
            #     print("Last frame window0 == first frame window1 →", overlap)

            # print("="*50)

            for key, value in zip(
                ['imu_inputs', 'pose_outputs', 'joint_outputs', 'tran_outputs'],
                [imu_input, pose, joint, tran]
            ):
                # data[key].extend(torch.split(value, data_len))
                splits = torch.split(value, data_len)

                # remove last if smaller than full window
                if splits[-1].shape[0] < data_len:
                    splits = splits[:-1]

                data[key].extend(splits)

            if not (self.evaluate or self.finetune):
                self._process_translation_data(joint, tran, foot, data_len, data)

    def _process_translation_data(self, joint, tran, foot, data_len, data):

        print("\nProcessing translation module")

        root_vel = torch.cat((torch.zeros(1, 3), tran[1:] - tran[:-1]))
        vel = torch.cat((torch.zeros(1, 24, 3), torch.diff(joint, dim=0)))
        vel[:, 0] = root_vel

        vel = vel * (datasets.fps / amass.vel_scale)

        vel_splits = torch.split(vel, data_len)

        print("Velocity windows created:", len(vel_splits))
        print("Velocity window shape:", vel_splits[0].shape)

        data['vel_outputs'].extend(vel_splits)
        data['foot_outputs'].extend(torch.split(foot, data_len))

    def __getitem__(self, idx):

        imu = self.data['imu_inputs'][idx].float()
        joint = self.data['joint_outputs'][idx].float()
        tran = self.data['tran_outputs'][idx].float()

        # print("\n" + "#"*60)
        # print("INSIDE __getitem__")
        # print("Index:", idx)
        # print("Raw pose window shape:",
        #       self.data['pose_outputs'][idx].shape)

        num_pred_joints = len(amass.pred_joints_set)

        pose = art.math.rotation_matrix_to_r6d(
            self.data['pose_outputs'][idx]
        ).reshape(-1, num_pred_joints, 6)[:, amass.pred_joints_set] \
         .reshape(-1, 6*num_pred_joints)

        # print("Pose AFTER r6d conversion shape:", pose.shape)
        # print("#"*60)

        if self.evaluate or self.finetune:
            return imu, pose, joint, tran

        vel = self.data['vel_outputs'][idx].float()
        contact = self.data['foot_outputs'][idx].float()

        return imu, pose, joint, tran, vel, contact

    def __len__(self):
        return len(self.data['imu_inputs'])
    




def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)


class PoseDataModule(L.LightningDataModule):
    def __init__(self, finetune: str = None):
        super().__init__()
        self.finetune = finetune
        self.hypers = finetune_hypers if self.finetune else train_hypers

    def setup(self, stage: str):
        if stage == 'fit':
            dataset = PoseDataset(fold='train', finetune=self.finetune)
            train_size = int(0.9 * len(dataset))
            val_size = len(dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        elif stage == 'test':
            self.test_dataset = PoseDataset(fold='test', finetune=self.finetune)

    def _dataloader(self, dataset):
        return DataLoader(
            dataset, 
            batch_size=self.hypers.batch_size, 
            collate_fn=pad_seq, 
            num_workers=0, #self.hypers.num_workers, 
            shuffle=True, 
            drop_last=True
        )

    def train_dataloader(self):
        return self._dataloader(self.train_dataset)

    def val_dataloader(self):
        return self._dataloader(self.val_dataset)

    def test_dataloader(self):
        return self._dataloader(self.test_dataset)

Total pose windows stored: 446112
Total training batches: 1568


In [7]:
# Create data module
datamodule = PoseDataModule(finetune='dip')

# IMPORTANT: must call setup
datamodule.setup(stage='fit')

# Get train loader
train_loader = datamodule.train_dataloader()

print("\nTotal training batches:", len(train_loader))



Loading datasets for TRAIN mode
Datasets: ['dip_train.pt']



  0%|          | 0/1 [00:00<?, ?it/s]


--------------------------------------------------
NEW SEQUENCE
Original pose shape from file: torch.Size([6883, 24, 3, 3])
Pose after forward kinematics: torch.Size([6883, 24, 3, 3])

Processing combo: lw_rp_h

Processing combo: rw_rp_h

Processing combo: lw_lp_h

Processing combo: rw_lp_h

Processing combo: lw_lp

Processing combo: lw_rp

Processing combo: rw_lp

Processing combo: rw_rp

Processing combo: lp_h

Processing combo: rp_h

Processing combo: lp

Processing combo: rp

--------------------------------------------------
NEW SEQUENCE
Original pose shape from file: torch.Size([3963, 24, 3, 3])
Pose after forward kinematics: torch.Size([3963, 24, 3, 3])

Processing combo: lw_rp_h

Processing combo: rw_rp_h

Processing combo: lw_lp_h

Processing combo: rw_lp_h

Processing combo: lw_lp

Processing combo: lw_rp

Processing combo: rw_lp

Processing combo: rw_rp

Processing combo: lp_h

Processing combo: rp_h

Processing combo: lp

Processing combo: rp

-----------------------------

C:\Users\degu03\AppData\Local\Temp\ipykernel_15216\1448404695.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.load(data_folder / data_file)


Pose after forward kinematics: torch.Size([4751, 24, 3, 3])

Processing combo: lw_rp_h

Processing combo: rw_rp_h

Processing combo: lw_lp_h

Processing combo: rw_lp_h

Processing combo: lw_lp

Processing combo: lw_rp

Processing combo: rw_lp

Processing combo: rw_rp

Processing combo: lp_h

Processing combo: rp_h

Processing combo: lp

Processing combo: rp

--------------------------------------------------
NEW SEQUENCE
Original pose shape from file: torch.Size([3204, 24, 3, 3])
Pose after forward kinematics: torch.Size([3204, 24, 3, 3])

Processing combo: lw_rp_h

Processing combo: rw_rp_h

Processing combo: lw_lp_h

Processing combo: rw_lp_h

Processing combo: lw_lp

Processing combo: lw_rp

Processing combo: rw_lp

Processing combo: rw_rp

Processing combo: lp_h

Processing combo: rp_h

Processing combo: lp

Processing combo: rp

--------------------------------------------------
NEW SEQUENCE
Original pose shape from file: torch.Size([3940, 24, 3, 3])
Pose after forward kinematics:

100%|██████████| 1/1 [00:00<00:00,  1.96it/s]


Processing combo: rw_rp_h

Processing combo: lw_lp_h

Processing combo: rw_lp_h

Processing combo: lw_lp

Processing combo: lw_rp

Processing combo: rw_lp

Processing combo: rw_rp

Processing combo: lp_h

Processing combo: rp_h

Processing combo: lp

Processing combo: rp

--------------------------------------------------
NEW SEQUENCE
Original pose shape from file: torch.Size([493, 24, 3, 3])
Pose after forward kinematics: torch.Size([493, 24, 3, 3])

Processing combo: lw_rp_h

Processing combo: rw_rp_h

Processing combo: lw_lp_h

Processing combo: rw_lp_h

Processing combo: lw_lp

Processing combo: lw_rp

Processing combo: rw_lp

Processing combo: rw_rp

Processing combo: lp_h

Processing combo: rp_h

Processing combo: lp

Processing combo: rp

--------------------------------------------------
NEW SEQUENCE
Original pose shape from file: torch.Size([2927, 24, 3, 3])
Pose after forward kinematics: torch.Size([2927, 24, 3, 3])

Processing combo: lw_rp_h

Processing combo: rw_rp_h

Proc

In [8]:
print("\nTotal training batches:", len(train_loader))


Total training batches: 334


In [9]:
sample = datamodule.train_dataset[0]

In [10]:
print("Sample loaded")

imu, pose, joint, tran = sample[:4]

print("IMU shape:", imu.shape)
print("Pose shape:", pose.shape)
print("Joint shape:", joint.shape)
print("Tran shape:", tran.shape)

Sample loaded
IMU shape: torch.Size([125, 60])
Pose shape: torch.Size([125, 144])
Joint shape: torch.Size([125, 24, 3])
Tran shape: torch.Size([125, 3])


In [11]:
batch = next(iter(train_loader))

(inputs, input_lengths), (outputs, output_lengths) = batch

print("\nBatch tensor shape (poses):", outputs['poses'].shape)
print("Original pose lengths:", output_lengths['poses'])

print("\nUnique lengths in batch:", set(output_lengths['poses']))
print("Max length in batch:", max(output_lengths['poses']))
print("Min length in batch:", min(output_lengths['poses']))


Batch tensor shape (poses): torch.Size([32, 125, 144])
Original pose lengths: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]

Unique lengths in batch: {125}
Max length in batch: 125
Min length in batch: 125


In [12]:
for (inputs, input_lengths), (outputs, output_lengths) in train_loader:
    # print("INPUT SHAPE:", inputs.shape)
    # print("POSES SHAPE:", outputs['poses'].shape)
    # print("JOINTS SHAPE:", outputs['joints'].shape)
    # print("TRANS SHAPE:", outputs['trans'].shape)
    print("INPUT LENGTHS:", input_lengths[:])  # first 5 lengths
    # break  # only look at first batch

INPUT LENGTHS: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]
INPUT LENGTHS: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]
INPUT LENGTHS: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]
INPUT LENGTHS: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]
INPUT LENGTHS: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]
INPUT LENGTHS: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125,

In [21]:

# Take one batch
batch = next(iter(train_loader))

print("\nBatch structure:")
print(type(batch))

(inputs, input_lengths), (outputs, output_lengths) = batch

print("\nInputs shape:", inputs.shape)
print("Input lengths:", input_lengths)

print("\nPose batch shape:", outputs['poses'].shape)
print("Pose lengths:", output_lengths['poses'])

print("\nJoint batch shape:", outputs['joints'].shape)
print("Trans batch shape:", outputs['trans'].shape)

if 'vels' in outputs:
    print("Velocity batch shape:", outputs['vels'].shape)

if 'foot_contacts' in outputs:
    print("Foot contact batch shape:", outputs['foot_contacts'].shape)


Batch structure:
<class 'tuple'>

Inputs shape: torch.Size([32, 125, 60])
Input lengths: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]

Pose batch shape: torch.Size([32, 125, 144])
Pose lengths: [125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125, 125]

Joint batch shape: torch.Size([32, 125, 24, 3])
Trans batch shape: torch.Size([32, 125, 3])


In [24]:
import torch
import torch.nn as nn
from typing import Sequence


# -------------------------------------------------
# Basic MLP Block
# -------------------------------------------------
class MLPBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# -------------------------------------------------
# Temporal Downsample (T -> T/2)
# -------------------------------------------------
class TemporalDown(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim * 2, dim)

    def forward(self, x):
        B, T, F = x.shape

        # pad if odd
        if T % 2 == 1:
            x = torch.nn.functional.pad(x, (0, 0, 0, 1))

        x = x.view(B, -1, 2, F)     # pair timesteps
        x = x.reshape(B, -1, 2 * F)  # concat
        x = self.proj(x)

        return x


# -------------------------------------------------
# Temporal Upsample (T -> T*2)
# -------------------------------------------------
class TemporalUp(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim * 2)

    def forward(self, x):
        B, T, F = x.shape
        x = self.proj(x)
        x = x.view(B, T * 2, F)
        return x


# -------------------------------------------------
#  Temporal Denoising UNet (Autoencoder)
# -------------------------------------------------
class TemporalDenoiseUNet(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        features: Sequence[int] = (128, 256, 256),
        dropout: float = 0.0,
    ):
        """
        feature_dim = input feature size F
        Output = reconstructed (B, T, F)
        """

        super().__init__()

        fea = list(features)

        # ---------------- Encoder ----------------
        self.enc0 = MLPBlock(feature_dim, fea[0], dropout)
        self.down1 = TemporalDown(fea[0])
        self.enc1 = MLPBlock(fea[0], fea[1], dropout)

        self.down2 = TemporalDown(fea[1])
        self.enc2 = MLPBlock(fea[1], fea[2], dropout)

        # ---------------- Decoder ----------------
        self.up2 = TemporalUp(fea[2])
        self.dec2 = MLPBlock(fea[2] + fea[1], fea[1], dropout)

        self.up1 = TemporalUp(fea[1])
        self.dec1 = MLPBlock(fea[1] + fea[0], fea[0], dropout)

        self.final = nn.Linear(fea[0], feature_dim)

    # -------------------------------------------------
    def forward(self, x):
        """
        x: (B, T, F)
        returns reconstructed x_hat: (B, T, F)
        """

        # -------- Encoder --------
        x0 = self.enc0(x)

        x1 = self.down1(x0)
        x1 = self.enc1(x1)

        x2 = self.down2(x1)
        x2 = self.enc2(x2)

        # -------- Decoder --------
        u2 = self.up2(x2)

        # pad if needed
        if u2.shape[1] != x1.shape[1]:
            diff = x1.shape[1] - u2.shape[1]
            u2 = torch.nn.functional.pad(u2, (0, 0, 0, diff))

        u2 = torch.cat([u2, x1], dim=-1)
        u2 = self.dec2(u2)

        u1 = self.up1(u2)

        if u1.shape[1] != x0.shape[1]:
            diff = x0.shape[1] - u1.shape[1]
            u1 = torch.nn.functional.pad(u1, (0, 0, 0, diff))

        u1 = torch.cat([u1, x0], dim=-1)
        u1 = self.dec1(u1)

        out = self.final(u1)

        return out

In [25]:
# model = TemporalDenoiseUNet(feature_dim=144)
# # out = model(outputs['poses'])
# # print(out.shape)

In [26]:
# x = outputs['poses']
# y = model(x)

# print("Input:", x.shape)
# print("Output:", y.shape)
# print("Difference:", (x - y).abs().mean())

Input: torch.Size([32, 125, 144])
Output: torch.Size([32, 125, 144])
Difference: tensor(0.4469, grad_fn=<MeanBackward0>)


In [31]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TemporalDenoiseUNet(feature_dim=144).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=1e-4
)

criterion = nn.MSELoss(reduction="none")

num_epochs = 100

for epoch in range(num_epochs):

    model.train()

    epoch_weighted_loss = 0.0
    total_frames = 0

    for batch in train_loader:

        (inputs, input_lengths), (outputs, output_lengths) = batch

        # -------------------------------------------------
        # Get Pose Data
        # -------------------------------------------------
        x = outputs["poses"].to(device)
        lengths = torch.as_tensor(
            output_lengths["poses"],
            device=device
        )

        B, T, F = x.shape

        # -------------------------------------------------
        # Optional: Add Noise (Denoising Training)
        # -------------------------------------------------
        noise = torch.randn_like(x) * 0.01
        noisy_x = x + noise

        # -------------------------------------------------
        # Forward
        # -------------------------------------------------
        pred = model(noisy_x)

        loss_matrix = criterion(pred, x)

        # -------------------------------------------------
        # Mask Padding Frames
        # -------------------------------------------------
        mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1).float()

        masked_loss = (loss_matrix * mask).sum()
        num_valid = mask.sum()

        if num_valid > 0:
            loss = masked_loss / num_valid
        else:
            loss = torch.tensor(
                0.0,
                device=device,
                requires_grad=True
            )

        # -------------------------------------------------
        # Backprop
        # -------------------------------------------------
        optimizer.zero_grad()
        loss.backward()

        #  Gradient Clipping (VERY IMPORTANT)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # -------------------------------------------------
        # Logging
        # -------------------------------------------------
        with torch.no_grad():
            grad_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    grad_norm += p.grad.data.norm(2).item() ** 2
            grad_norm = grad_norm ** 0.5

        print(
            f"Batch Loss: {loss.item():.6f} | "
            f"Grad Norm: {grad_norm:.4f}"
        )

        epoch_weighted_loss += masked_loss.item()
        total_frames += num_valid.item()

    # -------------------------------------------------
    # Epoch Summary
    # -------------------------------------------------
    if total_frames > 0:
        epoch_loss = epoch_weighted_loss / total_frames
    else:
        epoch_loss = 0.0

    print("===================================")
    print(f"Epoch {epoch}")
    print(f"Epoch Loss: {epoch_loss:.6f}")
    print("===================================")
torch.save(model.state_dict(), f"temporal_unet_epoch.pth")

Batch Loss: 48.388863 | Grad Norm: 1.0000
Batch Loss: 48.284039 | Grad Norm: 1.0000
Batch Loss: 48.380413 | Grad Norm: 1.0000
Batch Loss: 48.229492 | Grad Norm: 1.0000
Batch Loss: 48.105244 | Grad Norm: 1.0000
Batch Loss: 48.183804 | Grad Norm: 1.0000
Batch Loss: 48.201324 | Grad Norm: 1.0000
Batch Loss: 48.078686 | Grad Norm: 1.0000
Batch Loss: 48.113266 | Grad Norm: 1.0000
Batch Loss: 48.102516 | Grad Norm: 1.0000
Batch Loss: 47.987179 | Grad Norm: 1.0000
Batch Loss: 47.924068 | Grad Norm: 1.0000
Batch Loss: 47.901150 | Grad Norm: 1.0000
Batch Loss: 47.838558 | Grad Norm: 1.0000
Batch Loss: 47.661617 | Grad Norm: 1.0000
Batch Loss: 47.735725 | Grad Norm: 1.0000
Batch Loss: 47.699799 | Grad Norm: 1.0000
Batch Loss: 47.590363 | Grad Norm: 1.0000
Batch Loss: 47.769878 | Grad Norm: 1.0000
Batch Loss: 47.593227 | Grad Norm: 1.0000
Batch Loss: 47.520157 | Grad Norm: 1.0000
Batch Loss: 47.587765 | Grad Norm: 1.0000
Batch Loss: 47.482571 | Grad Norm: 1.0000
Batch Loss: 47.523483 | Grad Norm:

In [32]:
val_loader = datamodule.val_dataloader()

In [33]:
model.eval()

val_weighted_loss = 0.0
val_total_frames = 0

with torch.no_grad():
    for batch in val_loader:

        (inputs, input_lengths), (outputs, output_lengths) = batch

        x = outputs["poses"].to(device)
        lengths = torch.as_tensor(
            output_lengths["poses"],
            device=device
        )

        B, T, F = x.shape

        #  NO noise during validation
        pred = model(x)

        loss_matrix = nn.functional.mse_loss(
            pred,
            x,
            reduction="none"
        )

        mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1).float()

        masked_loss = (loss_matrix * mask).sum()
        num_valid = mask.sum()

        val_weighted_loss += masked_loss.item()
        val_total_frames += num_valid.item()

if val_total_frames > 0:
    val_loss = val_weighted_loss / val_total_frames
else:
    val_loss = 0.0

print(f"Validation Loss: {val_loss:.6f}")

Validation Loss: 0.109620


In [34]:
from articulate.evaluator import MeanPerJointErrorEvaluator
from articulate.math import RotationRepresentation

In [35]:
mpj_eval = MeanPerJointErrorEvaluator(
    official_model_file=paths.smpl_file,
    rep=RotationRepresentation.ROTATION_MATRIX,
    device=device
)